# FARK-SIM Agent Initialisation Pipeline using local model

Stratified sampling from Nemotron-Personas-Singapore.

Stratified by: gender × marital_status (rel_status).

Target: **200 agents (104F, 96M)** fitted to M&P 2021 proportions.

Downstream LLM steps add:
- `financial_security_score` (1–5, LLM-inferred)
- `relationship_status` (Single → Single/Dating, LLM-inferred)


Step 1: Load Dataset, inspect

In [1]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import json
import uuid

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ─────────────────────────────────────────────
# 1. LOAD AND INSPECT
# ─────────────────────────────────────────────

ds = load_dataset("nvidia/Nemotron-Personas-Singapore")
df = pd.DataFrame(ds["train"])

print("=== Raw Dataset Columns ===")
print(df.columns.tolist())
print(f"\nTotal records: {len(df)}")
print("\nSample row:")
print(df.iloc[0].to_dict())

c:\Users\chong\OneDrive - Singapore Management University\research\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== Raw Dataset Columns ===
['uuid', 'professional_persona', 'sports_persona', 'arts_persona', 'travel_persona', 'culinary_persona', 'persona', 'cultural_background', 'skills_and_expertise', 'skills_and_expertise_list', 'hobbies_and_interests', 'hobbies_and_interests_list', 'career_goals_and_ambitions', 'sex', 'age', 'marital_status', 'education_level', 'occupation', 'industry', 'planning_area', 'country']

Total records: 148000

Sample row:
{'uuid': 'd792702ec018494e8e49c69120759408', 'professional_persona': 'Yi Peng Yong, known as Danelle, a 20‑year‑old retail operations associate, leverages her methodical planning, bilingual communication, and data‑driven decision‑making to keep inventory flowing smoothly and nurture harmonious vendor relationships while adhering to a strict routine.', 'sports_persona': 'Yi Peng Yong, known as Danelle, enjoys playing golf at Marina Bay Golf Club, follows the Singapore Open and the PGA Tour, and fits short cardio sessions into her weekly routine to c

2. Normalise Fields 

Adjust field names here if Nemotron uses different names

Common Nemotron field names based on HuggingFace docs

In [2]:
FIELD_MAP = {
    "age":            "age",
    "gender":         "sex",                    # or "gender" — check your print above
    "rel_status":     "marital_status",
    "education":      "education_level",
    "occupation":     "occupation",
    "industry":       "industry",
    "planning_area":  "planning_area",
    "persona":        "general_persona",
    "cultural_bg":    "cultural_background",
    "hobbies":        "hobbies_and_interests",
    "career_goals":   "career_goals_and_ambitions",
}

# Rename to internal names for cleanliness
df = df.rename(columns={v: k for k, v in FIELD_MAP.items()})

cols = [
    "age",
    "gender",
    "rel_status",
    "education",
    "occupation",
    "industry",
    "planning_area",
    "persona",
    "cultural_bg",
    "hobbies",
    "career_goals"
]

df = df[cols].copy()

# Filter age 21–45
df = df[(df["age"] >= 21) & (df["age"] <= 45)].copy()

# Normalise gender
df["gender"] = df["gender"].str.lower().str.strip()
df = df[df["gender"].isin(["male", "female"])].copy()

# Normalise relationship status (dataset has single/married/divorced)
df["rel_status"] = df["rel_status"].str.lower().str.strip()
df = df[df["rel_status"].isin(["single", "married"])].copy()

print(f"\nAfter filtering (age 21-45, valid gender/rel): {len(df)} records")



After filtering (age 21-45, valid gender/rel): 62669 records


In [3]:
TARGETS = {
    ("male",   "single"):  50,
    ("female", "single"):  47,
    ("male",   "married"): 46,
    ("female", "married"): 57,
}
 
assert sum(TARGETS.values()) == 200, "Total must be 200"
assert sum(v for (g,_),v in TARGETS.items() if g == "male")   == 96,  "Male must be 96"
assert sum(v for (g,_),v in TARGETS.items() if g == "female") == 104, "Female must be 104"
 

4. STRATIFIED SAMPLING (age/education preserved as-is from Nemotron)

In [4]:
sampled_frames = []
sampling_log = {}
 
for (gender, rel), n in TARGETS.items():
    cell = df[(df["gender"] == gender) & (df["rel_status"] == rel)]
    available = len(cell)
 
    if available >= n:
        sample = cell.sample(n, random_state=RANDOM_SEED)
        method = "without_replacement"
    else:
        sample = cell.sample(n, replace=True, random_state=RANDOM_SEED)
        method = f"with_replacement (only {available} available)"
        print(f"WARN ({gender}, {rel}): needed {n}, had {available} -- sampled with replacement")
 
    sampling_log[(gender, rel)] = {"needed": n, "available": available, "method": method}
    sampled_frames.append(sample)
 
agents_df = pd.concat(sampled_frames).reset_index(drop=True)

5. ASSIGN IDs + NEUTRAL TPB BASELINE (Step 0)

In [5]:
 
agents_df["agent_id"] = [f"agent_{i:03d}" for i in range(1, len(agents_df) + 1)]

6. POST-HOC PLAUSIBILITY CHECK vs M&P 2021 Annex A

We did NOT fit age or education -- these are whatever Nemotron gave us. This block just reports them so you can note alignment/divergence.

In [6]:
print("\n=== Post-Hoc Check vs M&P 2021 Annex A ===")
print(f"\nTotal agents: {len(agents_df)}  (expect 200)")
 
print("\nGender x Marital (stratified -- should match targets exactly):")
print(pd.crosstab(agents_df["gender"], agents_df["rel_status"]).to_string())
 
print("\nRelationship status (after dating subdivision):")
print(agents_df["rel_status"].value_counts().to_string())
 
# Age -- M&P targets shown for reference (NOT fitted)
print("\nAge band -- SINGLE  [M&P ref: 21-25=30%, 26-30=34%, 31-35=18%, 36-40=10%, 41-45=9%]")
single = agents_df[agents_df["rel_status"] == "single"]
print((pd.cut(single["age"], [20,25,30,35,40,45],
              labels=["21-25","26-30","31-35","36-40","41-45"])
       .value_counts(normalize=True).sort_index()*100).round(1).to_string())
 
print("\nAge band -- MARRIED [M&P ref: 21-25=1%, 26-30=9%, 31-35=24%, 36-40=30%, 41-45=37%]")
married = agents_df[agents_df["rel_status"] == "married"]
print((pd.cut(married["age"], [20,25,30,35,40,45],
              labels=["21-25","26-30","31-35","36-40","41-45"])
       .value_counts(normalize=True).sort_index()*100).round(1).to_string())
 
# Education -- M&P targets shown for reference (NOT fitted)
print("\nEducation -- SINGLE  [M&P ref: Secondary<=12%, Diploma/A=34%, Degree+=55%]")
print(single["education"].value_counts(normalize=True).mul(100).round(1).to_string())
 
print("\nEducation -- MARRIED [M&P ref: Secondary<=16%, Diploma/A=28%, Degree+=56%]")
print(married["education"].value_counts(normalize=True).mul(100).round(1).to_string())


=== Post-Hoc Check vs M&P 2021 Annex A ===

Total agents: 200  (expect 200)

Gender x Marital (stratified -- should match targets exactly):
rel_status  married  single
gender                     
female           57      47
male             46      50

Relationship status (after dating subdivision):
rel_status
married    103
single      97

Age band -- SINGLE  [M&P ref: 21-25=30%, 26-30=34%, 31-35=18%, 36-40=10%, 41-45=9%]
age
21-25    35.1
26-30    33.0
31-35    20.6
36-40     7.2
41-45     4.1

Age band -- MARRIED [M&P ref: 21-25=1%, 26-30=9%, 31-35=24%, 36-40=30%, 41-45=37%]
age
21-25     2.9
26-30     9.7
31-35    21.4
36-40    30.1
41-45    35.9

Education -- SINGLE  [M&P ref: Secondary<=12%, Diploma/A=34%, Degree+=55%]
education
University                       52.6
Post Secondary (Non-Tertiary)    14.4
Polytechnic                      11.3
Other Diploma                    11.3
Secondary                         7.2
Lower Secondary                   2.1
Primary                   

## 7. LLM-Inferred Attributes (OpenRouter, free models)

Generate two LLM-inferred fields per agent:

- `financial_security_score` (1–5) — inferred from education, occupation, industry, age, planning area
- `relationship_status` (Single / Dating) — for single agents only, inferred from persona, age, education, hobbies, cultural background, career goals

Reasoning is preserved alongside each score / label for traceability.

Requires `OPENROUTER_API_KEY` in `.env`. Uses free model `meta-llama/llama-3.3-70b-instruct:free`.

In [ ]:
import os
import json
import time
import requests
import openai  # Added this to prevent NameError
from dotenv import load_dotenv

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
LOCAL_LLM_URL = os.getenv("LOCAL_LLM_URL")

if not OPENROUTER_API_KEY:
    raise RuntimeError("Set OPENROUTER_API_KEY in .env before running this cell")

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL = "nvidia/nemotron-3-super-120b-a12b:free"

# Limit how many agents the LLM steps below process.
# Set TEST_N = 5  -> dry-run on first 5 agents (uses ~10 free-tier requests)
# Set TEST_N = None -> run full 200 agents (uses ~400 free-tier requests)
TEST_N = 5

# Free-tier note: typically ~20 req/min and ~50/day shared across all :free models
# (1000/day after a $10 OpenRouter credit top-up). 200 agents x 2 calls = 400.
# If you hit the daily cap, top up or swap MODEL to a paid one and re-run only the
# rows where the score / status is None / "ERROR".

def _strip_code_fences(text: str) -> str:
    text = text.strip()
    if text.startswith("```"):
        parts = text.split("```")
        if len(parts) >= 2:
            text = parts[1]
            if text.lower().startswith("json"):
                text = text[4:]
    return text.strip("` \n")

def call_openrouter(system_prompt: str, user_prompt: str,
                    max_retries: int = 3, request_delay: float = 3.0) -> dict:
    """
    Call OpenRouter and return parsed JSON. Handles:
      - 429 rate-limit (waits then retries)
      - transient network errors (exponential backoff)
      - free-model JSON cosmetics (code fences, leading prose)

    request_delay sleeps AFTER a successful call to stay under free-tier RPM.
    """
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        "response_format": {"type": "json_object"},
        "temperature": 0.2,
    }

    last_err = None
    for attempt in range(max_retries):
        try:
            r = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=60)

            if r.status_code == 429:
                wait = 10 * (attempt + 1)
                print(f"    [429] rate limited — waiting {wait}s")
                time.sleep(wait)
                continue

            r.raise_for_status()
            content = r.json()["choices"][0]["message"]["content"]
            parsed = json.loads(_strip_code_fences(content))
            time.sleep(request_delay)
            return parsed

        except (requests.RequestException, json.JSONDecodeError, KeyError) as e:
            last_err = e
            wait = 2 ** attempt
            print(f"    attempt {attempt+1} failed: {type(e).__name__}: {e}. Retrying in {wait}s")
            time.sleep(wait)

    raise RuntimeError(f"call_openrouter exhausted retries; last error: {last_err}")


# Point the client to your local GPU backend running on port 49172
local_client = openai.OpenAI(
    base_url=LOCAL_LLM_URL,
    api_key="not-needed-locally"
)

def call_local_llm(system_prompt: str, user_prompt: str, max_retries: int = 3) -> dict:
    """
    Queries the local L40S GPU vLLM server instead of OpenRouter.
    Forces JSON output to perfectly match the FARK-SIM expectation.
    """
    for attempt in range(max_retries):
        try:
            response = local_client.chat.completions.create(
                model="meta-llama/Meta-Llama-3.1-8B-Instruct",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                # This guarantees the local model formats it as a valid Python/JSON dictionary
                extra_body={"guided_json": {
                    "type": "object",
                    "properties": {
                        "score": {"type": "integer"},
                        "status": {"type": "string"},
                        "reasoning": {"type": "string"}
                    }
                }},
                temperature=0.2
            )
            
            content = response.choices[0].message.content
            return json.loads(content)
            
        except Exception as e:
            print(f"    Attempt {attempt+1} failed locally: {e}")
            time.sleep(1)
            
    raise RuntimeError("Local LLM exhausted retries.")

# Sanity ping (uses 1 request from your daily quota)
_test = call_openrouter(
    "You respond only with valid JSON.",
    'Respond with exactly: {"status": "ok"}'
)
print("Sanity check:", _test)
print(f"TEST_N = {TEST_N}  (None = run all agents)")

Sanity check: {'status': 'ok'}
TEST_N = 5  (None = run all agents)


### 7a. Financial Security Score (1–5)

LLM infers SES band from education, occupation, industry, age, and planning area. Reasoning is preserved for traceability.

In [ ]:
FIN_SYSTEM_PROMPT = """You are a Singapore socio-economic analyst. Given an adult's demographic profile, estimate their financial security on a 1-5 scale where:
1 = Low (financially struggling, minimal savings/assets)
2 = Lower-middle
3 = Middle (stable, modest savings)
4 = Upper-middle
5 = Upper (high income, substantial assets)

Use cues: occupation seniority, industry pay norms in Singapore, education level, Singapore planning area (HDB heartland vs private estate / mature vs non-mature), and age (career stage). Be calibrated — most working adults are 2-4, not 5.

Respond ONLY with valid JSON: {"score": <int 1-5>, "reasoning": "<one or two sentences>"}"""

def build_fin_prompt(row) -> str:
    return (
        f"Age: {row['age']}\n"
        f"Gender: {row['gender']}\n"
        f"Education level: {row['education']}\n"
        f"Occupation: {row['occupation']}\n"
        f"Industry: {row.get('industry') or 'Not specified'}\n"
        f"Singapore planning area: {row['planning_area']}"
    )

# Initialise output cols (so partial test runs leave None for skipped agents)
agents_df["financial_security_score"]     = None
agents_df["financial_security_reasoning"] = None

target_df = agents_df.head(TEST_N) if TEST_N else agents_df
print(f"Running financial_security_score for {len(target_df)} agent(s)...\n")

for i, row in target_df.iterrows():
    print(f"[{i+1:3d}/{len(agents_df)}] {row['agent_id']}", end=" ")
    try:
        result = call_local_llm(FIN_SYSTEM_PROMPT, build_fin_prompt(row))
        score = int(result["score"])
        reasoning = str(result["reasoning"])
        if not 1 <= score <= 5:
            raise ValueError(f"score out of range: {score}")
    except Exception as e:
        print(f"FAILED -> {e}")
        score, reasoning = None, f"ERROR: {e}"
    else:
        print(f"-> {score}")
    agents_df.at[i, "financial_security_score"]     = score
    agents_df.at[i, "financial_security_reasoning"] = reasoning

print("\n=== Distribution of financial_security_score ===")
print(agents_df["financial_security_score"].value_counts(dropna=False))

Running financial_security_score for 5 agent(s)...

[  1/200] agent_001 -> 3
[  2/200] agent_002 -> 2
[  3/200] agent_003 -> 3
[  4/200] agent_004 -> 3
[  5/200] agent_005     attempt 1 failed: JSONDecodeError: Expecting value: line 147 column 1 (char 803). Retrying in 1s
-> 3

=== Distribution of financial_security_score ===
financial_security_score
None    195
3         4
2         1
Name: count, dtype: int64


### 7b. Relationship Status (Single → Single / Dating)

Single agents are LLM-judged into `Single` or `Dating` from persona, age, education, career goals, hobbies, and cultural background. Married agents pass through unchanged with source `raw_marital_status`.

In [ ]:
REL_SYSTEM_PROMPT = """You are a behavioural researcher reading an adult's profile to judge whether they are most plausibly:
- "Single": not in any romantic relationship right now
- "Dating": in a non-marital romantic relationship (boyfriend/girlfriend, partner, courtship)

Weigh cues such as: career intensity, lifestyle and hobbies, age, persona narrative (any mention of partners, dating, social life, family orientation), cultural and educational background. Do not default to one label; reason from the specific profile.

Respond ONLY with valid JSON: {"status": "Single" | "Dating", "reasoning": "<one or two sentences>"}"""

def build_rel_prompt(row) -> str:
    return (
        f"Age: {row['age']}\n"
        f"Gender: {row['gender']}\n"
        f"Education level: {row['education']}\n"
        f"Occupation: {row['occupation']}\n"
        f"Persona: {row['persona']}\n"
        f"Cultural background: {row['cultural_bg']}\n"
        f"Hobbies and interests: {row['hobbies']}\n"
        f"Career goals: {row['career_goals']}"
    )

# Initialise output cols (so partial test runs leave None for skipped agents)
agents_df["relationship_status"]           = None
agents_df["relationship_status_source"]    = None
agents_df["relationship_status_reasoning"] = None

target_df = agents_df.head(TEST_N) if TEST_N else agents_df
print(f"Running relationship_status for {len(target_df)} agent(s) "
      f"(LLM only called for Single agents)...\n")

for i, row in target_df.iterrows():
    if row["rel_status"] != "single":
        agents_df.at[i, "relationship_status"]        = "Married"
        agents_df.at[i, "relationship_status_source"] = "raw_marital_status"
        agents_df.at[i, "relationship_status_reasoning"] = None
        continue

    print(f"[{i+1:3d}/{len(agents_df)}] {row['agent_id']}", end=" ")
    try:
        result = call_local_llm(REL_SYSTEM_PROMPT, build_rel_prompt(row))
        status = str(result["status"]).strip().title()
        if status not in {"Single", "Dating"}:
            raise ValueError(f"unexpected status: {status}")
        reasoning = str(result["reasoning"])
    except Exception as e:
        print(f"FAILED -> {e}")
        status, reasoning = "Single", f"ERROR: {e}"  # safe fallback
    else:
        print(f"-> {status}")

    agents_df.at[i, "relationship_status"]           = status
    agents_df.at[i, "relationship_status_source"]    = "llm_imputed"
    agents_df.at[i, "relationship_status_reasoning"] = reasoning

print("\n=== Distribution of relationship_status ===")
print(agents_df["relationship_status"].value_counts(dropna=False))

Running relationship_status for 5 agent(s) (LLM only called for Single agents)...

[  1/200] agent_001 -> Single
[  2/200] agent_002 -> Single
[  3/200] agent_003 -> Single
[  4/200] agent_004 -> Single
[  5/200] agent_005 -> Single

=== Distribution of relationship_status ===
relationship_status
None      195
Single      5
Name: count, dtype: int64


In [14]:
# print agents_df score and reasoning for llm derived fields
print("\n=== Sample of LLM-derived relationship_status with reasoning ===")
print(agents_df[["agent_id", "relationship_status", "relationship_status_reasoning"]]
      .head(TEST_N).to_string(index=False)) 

# print agents_df score and reasoning for financial_security_score
print("\n=== Sample of LLM-derived financial_security_score with reasoning ===")
print(agents_df[["agent_id", "financial_security_score", "financial_security_reasoning"]]
      .head(TEST_N).to_string(index=False))


=== Sample of LLM-derived relationship_status with reasoning ===
 agent_id relationship_status                                                                                                                                                                                                                         relationship_status_reasoning
agent_001              Single The profile contains no mention of a romantic partner, dating activity, or relationship-oriented social life; his described routines focus on career, fitness, hobbies with friends, and family traditions, which align more with a single lifestyle.
agent_002              Single                                                                                                    The profile describes close friendships and future family aspirations but contains no indication of a current romantic partner or dating activity.
agent_003              Single                                               The profile emphasizes solitar

## 8. Export to `agents_initialised.json`

In [ ]:
import json

def row_to_agent(row):
    fin_score = row.get("financial_security_score")
    return {
        "agent_id":               row["agent_id"],
        "age":                    int(row["age"]),
        "gender":                 str(row["gender"]).title(),

        # Raw marital_status from Nemotron (Single / Married)
        "marital_status":         str(row["rel_status"]).title(),

        # LLM-derived relationship status (Single / Dating / Married)
        "relationship_status":            row.get("relationship_status"),
        "relationship_status_source":     row.get("relationship_status_source"),
        "relationship_status_reasoning":  row.get("relationship_status_reasoning"),

        "education":              row.get("education"),
        "occupation":              row.get("occupation"),
        "industry":                row.get("industry"),
        "planning_area":           row.get("planning_area"),

        # LLM-inferred SES band (1-5) with reasoning
        "financial_security_score":     (int(fin_score) if pd.notna(fin_score) else None),
        "financial_security_reasoning": row.get("financial_security_reasoning"),

        "general_persona":         row.get("persona"),
        "cultural_background":     row.get("cultural_bg"),
        "hobbies_and_interests":   row.get("hobbies"),
        "career_goals":            row.get("career_goals"),

        # TPB belief state -- neutral baseline (updated after seed memories)
        "belief_state": {
            "attitude_score":           3,
            "subjective_norm_score":    3,
            "pbc_score":                3,
            "fertility_intention_dist": None,
        },

        # Memory stream -- populated in next pipeline step
        "memory_stream": [],
    }

agents = [row_to_agent(row) for _, row in agents_df.iterrows()]

with open("agents_llama_initialised.json", "w", encoding="utf-8") as f:
    json.dump(agents, f, indent=2, ensure_ascii=False)

print(f"\nSaved {len(agents)} agents to agents_initialised.json")
print("Next step: generate seed memories per agent")


Saved 200 agents to agents_initialised.json
Next step: generate seed memories per agent
